# Workflow for simulation of droplet spreading

TODO: 
1) Check if final box size is correct (drop + substrate)
2) Check MD simulations at the end
3) Possibly add POST-PROCESSING

Either run `source ...` before opening the notebook, or define a variable for the GROMACS bin:

```gmx_bin = "<path-to-gmx-bin>"```

and pass it to the `topology_compiler` functions [LF: topology compiler is not used at the moment].

**NB** For this example, you may want to use the GROMACS fork with the flow binning code.

Import utility libraries and set the variable for the work directory.

**NB** run the `workdir` cell only once!

In [ ]:
import sys  
sys.path.insert(1, '../libraries/')

In [ ]:
from droplet_spreading import *

In [ ]:
workdir = os.getcwd()
print("Work directory:",workdir)

Set the following parameters for fluid and solid substrate

In [ ]:
n_atoms_mol_ZrO2 = 3 # number of atoms per molecule of ZrO2
n_atoms_mol_liq = 9 # number of atoms per molecule of R1234ze(E) or R1233zd(E)
fluid_name = "HFO-1234zeE"

In [ ]:
# Calibrating the initial droplet dimensions.
# Assuming a contact angle of 10 degrees, the fully spread droplet needs
# not to touch the periodic boundaries.

lx_0 = 5.26800
nbox_x = 15
Lx = lx_0*nbox_x
R0 = 0.5*(nbox_x*lx_0)-5.0
theta_0 = np.deg2rad(10)
r0 = FUN_RADIUS(theta_0,R0)
print("r0 =",r0,"nm")
print("Lx =",Lx,"nm")

## Solid substrate

In [ ]:
# Extending the solid substrate slab to create the surface
extend_substrate("zirconia.gro",nbox_x)

In [ ]:
!ls

In [ ]:
!vmd zirconia-ext.gro

In [ ]:
# LF: Determine current number of substrate molecules for topology

#n_atoms_mol_ZrO2 = 3 # number of atoms per molecule of ZrO2

with open("zirconia-ext.gro") as fp:
    for i, line in enumerate(fp):
        if i == 1:
            nmol_solid = int(line) // n_atoms_mol_ZrO2 # number of molecules of ZrO2 for topology update
            break


print(nmol_solid)

## Droplet

In [ ]:
# Solvating an empty box, from which the droplet will be carved.
# NB if the box is large, the number of atoms can be huge!
# Consider solvating a smaller box, and then resize it to match the length of the surface slab.
solvate_empty_box("box-HFO-1234zeE.gro", "empty.gro", "solvated.gro")

In [ ]:
!ls

In [ ]:
!vmd solvated.gro

In [ ]:
# Defining a lambda to 'carve' a cylinder of liquid out of the solvated box

cx = 0.5*Lx
cz = 3.5570000+6.654985089389049+0.3 # Sub. height + drop. radius + buffer
carve_condition = lambda x, y, z : ((x-cx)*(x-cx)+(z-cz)*(z-cz))<=(r0*r0)

In [ ]:
# Carving a mask according to the conditions defined above

# The number of atoms per molecules cannot be inferred, so it needs to be passed as input.
# TODO: create a wrapper that stores this type of information
#n_atoms_mol_liq = 9 # number of atoms per molecule of R1234ze(E) or R1233zd(E)

carve_gro("solvated.gro",n_atoms_mol_liq,carve_condition)

In [ ]:
!ls

In [ ]:
!vmd solvated-carved.gro

In [ ]:
# LF: Determine current number of liquid molecules for topology
# NB this is only liquid. in the topology we insert total fluid molecules (vapor + liquid)

#n_atoms_mol_liq = 9 # number of atoms per molecule of R1234ze(E) or R1233zd(E)

with open("solvated-carved.gro") as fp:
    for i, line in enumerate(fp):
        if i == 1:
            nmol_liq_only = int(line) // n_atoms_mol_liq # number of molecules of ZrO2 for topology update
            break


print(nmol_liq_only)

### Saturation with vapor

Using `gmx insert-molecule` one can also saturate the droplet+substrate box with vapour. This should ease equilibration, but then vapour molecules close to the $z$
  periodic boundaries needs to be "carved away":

In [ ]:
!gmx help insert-molecules

In [ ]:
import numpy as np
A = 79.02000*23.50389-np.pi*(r0**2)
rho_A = 0.77
nvap = int(A*rho_A)
print("nvap =",nvap)

In [ ]:
# LF: moved HFO-1234zeE.gro in same folder to be independent of other example
# LF: 1322 may be replaced with nvap
!gmx insert-molecules -f solvated-carved.gro -ci ../HFO-1234zeE.gro -o carved-saturated.gro -nmol 1322

In [ ]:
zupp = 21.40389
zlow = 3.5570000+0.3
carve_condition = lambda x, y, z : (z>zlow)*(z<zupp)

# The input files comes from a NVT simulation of the droplet in its vapour.
# Carving makes space for the surface and the reflective walls at the periodic edges.
# carve_gro("md-droplet/nvt.gro",n_atom_per_mol,carve_condition,output_file="carved-temp.gro")

carve_gro("carved-saturated.gro",n_atoms_mol_liq,carve_condition,output_file="carved-temp.gro")

In [ ]:
!ls

In [ ]:
!vmd carved-temp.gro

In [ ]:
# LF: Determine current number of fluid (liquid + vapor) molecules for topology

#n_atoms_mol_liq = 9 # number of atoms per molecule of R1234ze(E) or R1233zd(E)

with open("carved-temp.gro") as fp:
    for i, line in enumerate(fp):
        if i == 1:
            nmol_fluid = int(line) // n_atoms_mol_liq # number of molecules of ZrO2 for topology update
            break

print(nmol_fluid)

## Three-phase system setup

Here the droplet is merged with the substrate to create the configuration for droplet spreading, and the topology file is finally updated.

The box size corresponds to the liquid-vapor box size already carved. [check]

Run only once.

In [ ]:
!cp zirconia-ext.gro edit-solid.gro

In [ ]:
n_atoms_tot = nmol_fluid*n_atoms_mol_liq + nmol_solid*n_atoms_mol_ZrO2

with open("edit-solid.gro", "r") as fp:
    rows_s = fp.readlines()

rows_s[1] = str(n_atoms_tot) + "\n"

with open("edit-solid.gro", "w") as fp:
    fp.writelines(rows_s[:-1]) # delete last line with box
    
print(n_atoms_tot)

In [ ]:
# copy the fluid .gro
!cp carved-temp.gro edit-fluid.gro

with open("edit-fluid.gro", "r") as fp:
    rows_f = fp.readlines()

with open("edit-fluid.gro", "w") as fp:
    fp.writelines(rows_f[2:]) #delete lines 1-2

In [ ]:
# Fluid and solid are put together
!cat edit-solid.gro edit-fluid.gro > three-phase.gro

In [ ]:
# Final box check
with open("three-phase.gro", "r") as fp:
    rows_3p = fp.readlines()
    
box_line = rows_3p[-1]
print(box_line)

In [ ]:
# check if everything is ok
!vmd three-phase.gro

Now the topology file can be update.

NB If you run this tutorial multiple times, remember to delete the last lines of the topology file.

In [ ]:
# fluid_name = "HFO-1234zeE"

In [ ]:
with open("topology-droplet.top", "a") as fp :
    fp.write("\n"+"[ system ]"+"\n")
    fp.write("DropSpread"+"\n")
    fp.write("\n"+"[ molecules ]"+"\n")
    fp.write("ZrO2 "+str(nmol_solid)+"\n")
    fp.write(fluid_name+" "+str(nmol_fluid)+"\n")

Let us look at the final `topology-droplet.top`:

In [ ]:
!cat topology-droplet.top

In [ ]:
### Manipulation (done: check previous steps)

# Here are a few steps that still need to be implemented in the Python wrapper...

# Using `cat`, one can quickly merge the droplet to the substrate, but it's a bit boring since a few lines needs to be removed (e.g. with `vim`) before merging and the number of atoms need to be updated after merging.

# Using `gmx insert-molecule` one can also saturate the droplet+substrate box with vapour. This should ease equilibration, but then vapour molecules close to the $z$ periodic boundaries needs to be "carved away":

## Molecular dynamics

### Energy minimization

In [ ]:
!gmx grompp -p topology-droplet.top -c three-phase.gro -r three-phase.gro -f STEEP/sd.mdp -o STEEP/system.tpr

In [ ]:
 %cd {workdir}/STEEP
!gmx mdrun -v -s system.tpr
 %cd {workdir}

### NVT simulation with binning

For NVT simulation the initial configuration is the output of energy minimization, while restraints are given by the original configuration (`three-phase.gro`).

In [ ]:
!gmx grompp -p topology-droplet.top -c STEEP/confout.gro -r three-phase.gro -f NVT/nvt.mdp -o NVT/system.tpr

In [ ]:
 %cd {workdir}/NVT
!gmx mdrun -v -s system.tpr
 %cd {workdir}